# Indicadores – Pesca e Aquicultura (RESEX/UCs)

Execute as células em ordem (`Shift+Enter`) ou use **Executar Tudo**.
Só a célula 1 precisa ser editada: nomes dos arquivos e critérios.

## 1. Configuração (edite aqui)

In [90]:
# Arquivo na mesma pasta deste caderno
ARQUIVO = "consolidado_dasdosGerais_181126.xlsx"                    # seu arquivo com as 3 abas
ABA_DADOS_GERAIS = "Dados Gerais"          # use None se não quiser usar esta aba
ABA_MEMBROS      = "Membros Familiares"
ABA_ATIVIDADES   = "Atividades Produtivas"
# (maiúsculas, acentos e singular/plural nos nomes das abas não importam)

# Critérios da aquicultura (coluna "Produto Atividade Matéria Prima")
PRODUTOS_AQUICULTURA = ["Peixes (piscicultura)", "Camarões (carcinicultura)", "Ostras (ostreicultura)" ]
# Texto procurado em "Outro Produto" (não diferencia maiúsculas; acha em qualquer posição)
# mude se forem considerar valores diferentes
TEXTOS_OUTRO_AQUICULTURA = ["PSICULTURA", "VIVEIRO DE TAMBAQUI"]

# Nomes das colunas (só mude se os cabeçalhos forem diferentes)
UC, FAM, INTEG, SEXO, RGP, OCUP, NASC = "Unidade Conservação", "Código Familiar", "Código Integrante", "Sexo", "RGP - Possui", "Principais Ocupações", "Data Nascimento"
ATV, PROD, OUTRO = "Principais Atividades Produtivas", "Produto Atividade Matéria Prima", "Outro Produto"
IDADE_MINIMA = 18
DATA_REFERENCIA = None  # será considerado o dia que executar esse notebook

# Tipos de Unidade de Conservação
UC_TYPES = {
    "ESEC": "Estação Ecológica",
    "REBIO": "Reserva Biológica",
    "PARNA": "Parque Nacional",
    "MONA": "Monumento Natural",
    "RVS": "Refúgio de Vida Silvestre",
    "APA": "Área de Proteção Ambiental",
    "ARIE": "Área de Relevante Interesse Ecológico",
    "FLONA": "Floresta Nacional",
    "RESEX": "Reserva Extrativista",
    "REFAU": "Reserva de Fauna",
    "RDS": "Reserva de Desenvolvimento Sustentável",
    "RPPN": "Reserva Particular do Patrimônio Natural",
}


## 2. Leitura e limpeza dos dados

### 2.1 Funções auxiliares

In [ ]:
import unicodedata
import pandas as pd

# variáveis que precisam ter sido definidas na seção 1 (Configuração)
REQUIRED_CONFIG = [
    "ARQUIVO", "ABA_DADOS_GERAIS", "ABA_MEMBROS", "ABA_ATIVIDADES",
    "UC", "FAM", "INTEG", "SEXO", "RGP", "OCUP", "NASC", "ATV", "PROD", "OUTRO",
    "IDADE_MINIMA", "DATA_REFERENCIA", "UC_TYPES",
]
missing_config = [name for name in REQUIRED_CONFIG if name not in globals()]
assert not missing_config, f"Execute a seção 1 (Configuração). Variáveis não definidas: {missing_config}"


# texto -> sem acentos, em letras minúsculas, sem espaços no início ou no fim, sem espaços repetidos
def sanitize_text(text):
    text_without_accents = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return " ".join(text_without_accents.lower().split())


# remove espaços nas extremidades e reduz espaços repetidos; opera apenas sobre os valores únicos
def clean_spaces(table, columns):
    for column in columns:
        values = table[column]
        cleaned_by_value = {value: " ".join(str(value).split()) for value in values.dropna().unique()}
        table[column] = values.map(cleaned_by_value)   # células vazias permanecem vazias
    return table


# idade em anos completos na data de referência; datas vazias ou inválidas resultam em NaN
def calculate_age(birth_dates, reference_date):
    birth_date = pd.to_datetime(birth_dates, dayfirst=True, errors="coerce", format="mixed")
    had_birthday = (birth_date.dt.month < reference_date.month) | (
        (birth_date.dt.month == reference_date.month) & (birth_date.dt.day <= reference_date.day)
    )
    return reference_date.year - birth_date.dt.year - (~had_birthday).astype(int)


# busca um texto dentro de uma coluna (case-insensitive)
def contains_text(text_column, searched_text):
    return text_column.fillna("").astype(str).str.contains(searched_text, case=False, regex=False)


# nomes por extenso dos tipos de UC, sanitizados, para comparar com o início do nome de cada UC
SANITIZED_UC_TYPE_NAMES = {
    acronym: sanitize_text(full_name) for acronym, full_name in UC_TYPES.items()
}


# descobre a sigla pelo início do nome da UC; devolve None se não reconhecer
def get_uc_type(uc_name):
    sanitized_uc_name = sanitize_text(uc_name)
    for acronym, sanitized_type_name in SANITIZED_UC_TYPE_NAMES.items():
        if sanitized_uc_name.startswith(sanitized_type_name):
            return acronym
    return None


# siglas presentes nas UCs de um grupo
def get_uc_types_present(uc_names):
    types_found = {get_uc_type(uc_name) for uc_name in pd.Series(uc_names).dropna().unique()}
    return [acronym for acronym in UC_TYPES if acronym in types_found]


# nomes de UC cujo tipo não consta no dicionário
def get_unrecognized_ucs(uc_names):
    return [uc_name for uc_name in pd.Series(uc_names).dropna().unique() if get_uc_type(uc_name) is None]


### 2.2 Leitura dos dados

In [ ]:
# nomes das colunas da configuração, no mesmo formato dos cabeçalhos sanitizados
UC, FAM, INTEG, SEXO, RGP, OCUP, ATV, PROD, OUTRO, NASC = [
    sanitize_text(column_name)
    for column_name in (UC, FAM, INTEG, SEXO, RGP, OCUP, ATV, PROD, OUTRO, NASC)
]

# o "calamine" lê arquivos .xlsx; caso não seja possível, utiliza o mecanismo padrão
try:
    excel_file = pd.ExcelFile(ARQUIVO, engine="calamine")
except (ImportError, ValueError):
    excel_file = pd.ExcelFile(ARQUIVO)
print("Abas encontradas:", excel_file.sheet_names)

# {nome da aba sanitizado: nome da aba original}, criado uma única vez
sheet_names_by_sanitized = {
    sanitize_text(sheet_name): sheet_name for sheet_name in excel_file.sheet_names
}


def find_sheet_name(requested_sheet_name):
    sanitized_name = sanitize_text(requested_sheet_name)
    assert sanitized_name in sheet_names_by_sanitized, (
        f"Aba '{requested_sheet_name}' não encontrada. Abas do arquivo: {excel_file.sheet_names}"
    )
    return sheet_names_by_sanitized[sanitized_name]


# lê APENAS as colunas necessárias de uma aba, com cabeçalhos sanitizados
def read_sheet(requested_sheet_name, needed_columns):
    table = pd.read_excel(
        excel_file,
        sheet_name=find_sheet_name(requested_sheet_name),
        usecols=lambda column: sanitize_text(column) in needed_columns,
    )
    table.columns = [sanitize_text(column) for column in table.columns]
    assert not table.columns.duplicated().any(), f"Cabeçalhos repetidos na aba '{requested_sheet_name}'"
    missing_columns = [column for column in needed_columns if column not in table.columns]
    assert not missing_columns, f"Colunas não encontradas na aba '{requested_sheet_name}': {missing_columns}"
    return table


gerais_columns = [FAM]
ativ_columns = [FAM, ATV, PROD, OUTRO]
memb_columns = [UC, FAM, INTEG, SEXO, RGP, OCUP, NASC]

gerais = clean_spaces(read_sheet(ABA_DADOS_GERAIS, gerais_columns), gerais_columns)
ativ = clean_spaces(read_sheet(ABA_ATIVIDADES, ativ_columns), ativ_columns)
memb = clean_spaces(read_sheet(ABA_MEMBROS, memb_columns), [UC, FAM, INTEG, SEXO, RGP, OCUP])

gerais = gerais.drop_duplicates()


### 2.3 Idade e resumo da leitura

In [ ]:
# idade em anos completos, na data de referência
reference_date = pd.Timestamp(DATA_REFERENCIA) if DATA_REFERENCIA else pd.Timestamp.today().normalize()

memb["age"] = calculate_age(memb[NASC], reference_date)
memb["is_adult"] = memb["age"] >= IDADE_MINIMA

uc_count = memb[UC].nunique()

unrecognized_ucs = get_unrecognized_ucs(memb[UC])
if unrecognized_ucs:
    print(f"Atenção: tipo de UC não reconhecido em {len(unrecognized_ucs)} UC(s): {unrecognized_ucs[:3]}")

print(f"Atividades produtivas: {len(ativ):,} linhas | {ativ[FAM].nunique():,} famílias")
print(f"Membros familiares:    {len(memb):,} pessoas | {memb[FAM].nunique():,} famílias")
print(f"Dados gerais:          {gerais[FAM].nunique():,} famílias")
print(f"Unidades de conservação (UCs): {uc_count}")
print(f"Data de referência para a idade: {reference_date:%d/%m/%Y}")


## 3. Pesca

In [ ]:
fam_pesca = set(ativ.loc[contains_text(ativ[ATV], "Atividade pesqueira"), FAM])

print(f"Famílias com atividade pesqueira:       {len(fam_pesca):,}")

# ---------- PESSOAS (aba Membros Familiares) ----------
# Pessoas com "Atividade pesqueira" na coluna Principais Ocupações (mesmo com outras ocupações juntas)
pesca_pessoas = memb[contains_text(memb[OCUP], "Atividade pesqueira") & memb["is_adult"]]

total_pesca = len(pesca_pessoas)

cont = pesca_pessoas[RGP].fillna("(em branco)").value_counts()
pesca_rgp_sim   = int(cont.get("Sim", 0))
pesca_rgp_nao   = int(cont.get("Não", 0))
pesca_rgp_nsi   = int(cont.get("Não sabe/Não informou", 0))
pesca_rgp_vazio = int(cont.get("(em branco)", 0))

print(f"\nAdultos com atividade pesqueira: {total_pesca:,}")
print(f"  RGP Sim:                 {pesca_rgp_sim:,}  ({pesca_rgp_sim/total_pesca:.1%})")
print(f"  RGP Não:                 {pesca_rgp_nao:,}  ({pesca_rgp_nao/total_pesca:.1%})")
print(f"  Não sabe/Não informou:   {pesca_rgp_nsi:,}  ({pesca_rgp_nsi/total_pesca:.1%})")
if pesca_rgp_vazio:
    print(f"  Em branco:               {pesca_rgp_vazio:,}  ({pesca_rgp_vazio/total_pesca:.1%})")

tab_pesca = pd.crosstab(pesca_pessoas[SEXO], pesca_pessoas[RGP], margins=True, margins_name="Total")
tab_pesca

## 4. Aquicultura

In [ ]:
# ---------- 1) FAMÍLIAS: aba Atividades Produtivas ----------
products = ativ[PROD].fillna("")

# a) produto na lista de aquicultura
cond_produto = products.isin(PRODUTOS_AQUICULTURA)

# b) produto = "Outro" E "Outro Produto" com um dos textos
found_text = pd.concat([contains_text(ativ[OUTRO], t) for t in TEXTOS_OUTRO_AQUICULTURA], axis=1).any(axis=1)
cond_outro = (products == "Outro") & found_text

fam_aqui = set(ativ.loc[cond_produto | cond_outro, FAM])      # códigos familiares únicos

print(f"Famílias com aquicultura: {len(fam_aqui):,}")

# ---------- 2) VERIFICAÇÃO: aba Membros Familiares ----------
# todas as pessoas cujo código familiar está na lista acima
memb_aqui = memb[memb[FAM].isin(fam_aqui) & memb["is_adult"]]
total_aqui = len(memb_aqui)

cont = memb_aqui[RGP].fillna("(em branco)").value_counts()
aqui_rgp_sim   = int(cont.get("Sim", 0))
aqui_rgp_nao   = int(cont.get("Não", 0))
aqui_rgp_nsi   = int(cont.get("Não sabe/Não informou", 0))
aqui_rgp_vazio = int(cont.get("(em branco)", 0))

print(f"\nAdultos, nessas famílias, na aquicultura: {total_aqui:,}")
if total_aqui:
    print(f"  RGP Sim:                 {aqui_rgp_sim:,}  ({aqui_rgp_sim/total_aqui:.1%})")
    print(f"  RGP Não:                 {aqui_rgp_nao:,}  ({aqui_rgp_nao/total_aqui:.1%})")
    print(f"  Não sabe/Não informou:   {aqui_rgp_nsi:,}  ({aqui_rgp_nsi/total_aqui:.1%})")
    if aqui_rgp_vazio:
        print(f"  Em branco:               {aqui_rgp_vazio:,}  ({aqui_rgp_vazio/total_aqui:.1%})")

tab_aqui = pd.crosstab(memb_aqui[SEXO], memb_aqui[RGP], margins=True, margins_name="Total")
tab_aqui

Famílias com aquicultura: 51

Adultos, nessas famílias, na aquicultura: 97
  RGP Sim:                 10  (10.3%)
  RGP Não:                 80  (82.5%)
  Não sabe/Não informou:   7  (7.2%)


rgp - possui,Não,Não sabe/Não informou,Sim,Total
sexo,,,,
Feminino,36,2,5,43
Masculino,44,5,5,54
Total,80,7,10,97


## 5. Quadro-resumo

In [ ]:
# conta as linhas de uma tabela de pessoas, opcionalmente filtradas por sexo e/ou RGP
def count_people(people, sex=None, rgp=None):
    selected = pd.Series(True, index=people.index)
    if sex:
        selected &= people[SEXO] == sex
    if rgp:
        selected &= people[RGP] == rgp
    return int(selected.sum())

summary_rows = [
    ("Total de famílias", gerais[FAM].nunique()),

    ("Famílias com Atividade pesqueira como atividade principal", len(fam_pesca)),
    ("Familiares com RGP (Sim) - Atividade pesqueira", count_people(pesca_pessoas, rgp="Sim")),
    ("Familiares sem RGP (Não) - Atividade pesqueira", count_people(pesca_pessoas, rgp="Não")),
    ("Mulheres na pesca", count_people(pesca_pessoas, "Feminino")),
    ("Mulheres na pesca com RGP", count_people(pesca_pessoas, "Feminino", "Sim")),
    ("Homens na pesca", count_people(pesca_pessoas, "Masculino")),
    ("Homens na pesca com RGP", count_people(pesca_pessoas, "Masculino", "Sim")),

    ("Famílias com Aquicultura como atividade principal", len(fam_aqui)),
    ("Familiares com RGP (Sim) - Aquicultura", count_people(memb_aqui, rgp="Sim")),
    ("Familiares sem RGP (Não) - Aquicultura", count_people(memb_aqui, rgp="Não")),
    ("Mulheres na aquicultura", count_people(memb_aqui, "Feminino")),
    ("Mulheres na aquicultura com RGP", count_people(memb_aqui, "Feminino", "Sim")),
    ("Homens na aquicultura", count_people(memb_aqui, "Masculino")),
    ("Homens na aquicultura com RGP", count_people(memb_aqui, "Masculino", "Sim")),
]
resumo = pd.DataFrame(summary_rows, columns=["Indicador", "Quantitativo"])
resumo

## 6. Gráficos

In [ ]:
import textwrap
import matplotlib.pyplot as plt

RGP_COLORS = {"Sim": "#2a9d8f", "Não": "#9aa5b1", "Não sabe/Não informou": "#e9c46a"}
SEXES = ["Feminino", "Masculino"]

# Famílias por atividade principal
families_by_activity = ativ.groupby(ATV)[FAM].nunique().sort_values()
families_by_activity.index = [label.split(" (")[0][:60] for label in families_by_activity.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(families_by_activity.index, families_by_activity.values, color="#264653")
ax.set_title(f"Famílias por atividade produtiva principal em \n{uc_count} Unidades de Conservação")
ax.set_xlabel("Nº de famílias")
for position, count in enumerate(families_by_activity.values):
    ax.text(count, position, f" {count:,}", va="center", fontsize=8)
plt.tight_layout(); plt.savefig("grafico_atividades.png", dpi=150); plt.show()


# título: o número de unidades vem do próprio grupo de pessoas do gráfico
def build_title(subject, uc_count, uc_types):
    types_text = ", ".join(uc_types)
    title = (
        f"Informações de acesso ao RGP por {subject}, por sexo, "
        f"no levantamento de famílias de {uc_count} "
        f"unidades de conservação federais ({types_text})"
    )
    return "\n".join(textwrap.wrap(title, width=60))


# Pessoas por sexo e RGP (pesca e aquicultura)
def barras_sexo_rgp(people, subject, file_name):
    title = build_title(subject, people[UC].nunique(), get_uc_types_present(people[UC]))

    people_by_sex_and_rgp = pd.crosstab(people[SEXO], people[RGP]).reindex(SEXES).fillna(0)

    fig, ax = plt.subplots(figsize=(7, 5))
    bar_bottoms = pd.Series(0, index=people_by_sex_and_rgp.index)
    for rgp_status in [status for status in RGP_COLORS if status in people_by_sex_and_rgp.columns]:
        ax.bar(people_by_sex_and_rgp.index, people_by_sex_and_rgp[rgp_status],
               bottom=bar_bottoms, label=f"RGP: {rgp_status}", color=RGP_COLORS[rgp_status])
        bar_bottoms += people_by_sex_and_rgp[rgp_status]

    for position, total in enumerate(bar_bottoms):
        ax.text(position, total, f"{int(total):,}", ha="center", va="bottom")

    ax.set_title(title, fontsize=10)
    ax.set_ylabel("Pessoas")
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(file_name, dpi=150); plt.show()


barras_sexo_rgp(pesca_pessoas, "pescadores e pescadoras artesanais", "grafico_pesca_sexo_rgp.png")
barras_sexo_rgp(memb_aqui, "aquicultores familiares", "grafico_aquicultura_sexo_rgp.png")

## 7. Exportar resultados para Excel

In [ ]:
with pd.ExcelWriter("resultados_indicadores.xlsx") as xw:
    resumo.to_excel(xw, sheet_name="Resumo", index=False)
    tab_pesca.to_excel(xw, sheet_name="Pesca sexo x RGP")
    tab_aqui.to_excel(xw, sheet_name="Aquicultura sexo x RGP")
print("Arquivo gerado: resultados_indicadores.xlsx (e os gráficos .png)")